# Formal Layer2R one-case regression

This runtime-only wrapper verifies CUDA, resolves the attached dataset, runs the formal preflight, and executes at most one pending locked case.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import torch

from src.pipelines.formal_layer2r import load_formal_config

CONFIG_PATH = Path("configs/layer2r_kaggle_one_case.json")
RUNNER_PATH = Path("experiments/run_formal_layer2r.py")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the formal one-case regression")
print("CUDA device:", torch.cuda.get_device_name(0))

with CONFIG_PATH.open() as stream:
    raw_config = json.load(stream)
if raw_config.get("max_new_cases") != 1:
    raise RuntimeError("One-case wrapper requires max_new_cases == 1")

config = load_formal_config(CONFIG_PATH)
print("Resolved dataset root:", config.raw_root)

command = [sys.executable, str(RUNNER_PATH), "--config", str(CONFIG_PATH)]
preflight = subprocess.run([*command, "--preflight"], check=False)
if preflight.returncode != 0:
    raise RuntimeError(f"Formal one-case preflight failed with exit code {preflight.returncode}")

execution = subprocess.run(command, check=False)
if execution.returncode != 0:
    raise RuntimeError(f"Formal one-case execution failed with exit code {execution.returncode}")
print("Formal Layer2R one-case regression completed; wrapper stopping.")
